In [1]:
import numpy as np
from dataclasses import dataclass
from typing import Tuple, Dict

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import KFold
from scipy.stats import norm

import torch
import torch.nn as nn
import torch.optim as optim

# ---------------------------------------------------------
# 1. Your data: 3D inputs and 1D outputs (maximisation)
# ---------------------------------------------------------
X_raw = np.array([
    [0.17152521, 0.34391687, 0.2487372],
    [0.24211446, 0.64407427, 0.27243281],
    [0.53490572, 0.39850092, 0.17338873],
    [0.49258141, 0.61159319, 0.34017639],
    [0.13462167, 0.21991724, 0.45820622],
    [0.34552327, 0.94135983, 0.26936348],
    [0.15183663, 0.43999062, 0.99088187],
    [0.64550284, 0.39714294, 0.91977134],
    [0.74691195, 0.28419631, 0.22629985],
    [0.17047699, 0.6970324, 0.14916943],
    [0.22054934, 0.29782524, 0.34355534],
    [0.66601366, 0.67198515, 0.2462953],
    [0.04680895, 0.23136024, 0.77061759],
    [0.60009728, 0.72513573, 0.06608864],
    [0.96599485, 0.86111969, 0.56682913],
    [1.065994, 1.041359, 1.090881],  # historical out-of-bounds
    [0.403482, 0.38217, 0.489363],
    [3.98350e-01, 1.00000e-06, 5.43642e-01],
    [0.962851, 0.987386, 0.040875],
    [0.504564, 0.348726, 0.601264],
    [0.265159, 0.286931, 0.413777],
    [0.403756, 0.381706, 0.489738]
])

y_raw = np.array([
    -0.1121222,  -0.08796286, -0.11141465, -0.03483531, -0.04800758,
    -0.11062091, -0.39892551, -0.11386851, -0.13146061, -0.09418956,
    -0.04694741, -0.10596504, -0.11804826, -0.03637783, -0.05675837,
    -0.769427956661122, -0.03310307977430594, -0.09333459499358941,
    -0.07627377706316849, -0.05678719487656195, -0.03492633073917894, -0.009136026447950633
])

DEVICE = torch.device("cpu")  # change to "cuda" if you have a GPU


# ---------------------------------------------------------
# Utility: bounds + rounding for submission
# ---------------------------------------------------------
def clip_to_bounds(X: np.ndarray, lower: np.ndarray, upper: np.ndarray) -> np.ndarray:
    return np.clip(X, lower, upper)


def round6(x: np.ndarray) -> np.ndarray:
    """Round to 6 decimals for submission."""
    return np.round(x.astype(float), 6)


# ---------------------------------------------------------
# 2. PyTorch MLP model
# ---------------------------------------------------------
class MLPRegressorTorch(nn.Module):
    def __init__(self, input_dim: int, hidden_sizes=(64, 64), dropout=0.15):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev_dim, h))
            layers.append(nn.ReLU())
            if dropout > 0.0:
                layers.append(nn.Dropout(dropout))
            prev_dim = h
        layers.append(nn.Linear(prev_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)


def _train_mlp(
    Xs: np.ndarray,
    ys: np.ndarray,
    hidden: Tuple[int, ...],
    dropout: float,
    lr: float,
    weight_decay: float,
    n_epochs: int,
    seed: int,
    tol: float = 1e-6,
    patience: int = 80,
):
    """Train with simple early stopping on training loss."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = MLPRegressorTorch(
        input_dim=Xs.shape[1],
        hidden_sizes=hidden,
        dropout=dropout
    ).to(DEVICE)

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    X_tensor = torch.from_numpy(Xs.astype(np.float32)).to(DEVICE)
    y_tensor = torch.from_numpy(ys.astype(np.float32)).view(-1, 1).to(DEVICE)

    best_loss = float("inf")
    bad = 0

    for _ in range(n_epochs):
        model.train()
        optimizer.zero_grad()
        preds = model(X_tensor)
        loss = criterion(preds, y_tensor)
        loss.backward()
        optimizer.step()

        l = float(loss.item())
        if best_loss - l > tol:
            best_loss = l
            bad = 0
        else:
            bad += 1
            if bad >= patience:
                break

    return model


# ---------------------------------------------------------
# 3. MC Dropout surrogate (robust y scaling)
# ---------------------------------------------------------
@dataclass
class MCDropoutSurrogate:
    hidden_layer_sizes: Tuple[int, ...] = (64, 64)
    dropout: float = 0.15
    n_epochs: int = 1400
    lr: float = 1e-3
    weight_decay: float = 1e-6
    random_state: int = 0
    n_mc_samples: int = 256
    robust_y: bool = True

    def __post_init__(self):
        self.model = None
        self.x_scaler = StandardScaler()
        self.y_scaler = RobustScaler() if self.robust_y else StandardScaler()

    def fit(self, X: np.ndarray, y: np.ndarray, lower: np.ndarray, upper: np.ndarray):
        Xc = clip_to_bounds(X, lower, upper)
        Xs = self.x_scaler.fit_transform(Xc)
        ys = self.y_scaler.fit_transform(y.reshape(-1, 1)).ravel()

        self.model = _train_mlp(
            Xs, ys,
            hidden=self.hidden_layer_sizes,
            dropout=self.dropout,
            lr=self.lr,
            weight_decay=self.weight_decay,
            n_epochs=self.n_epochs,
            seed=self.random_state,
        )

    def predict(self, X: np.ndarray, lower: np.ndarray, upper: np.ndarray, return_std: bool = False):
        Xc = clip_to_bounds(X, lower, upper)
        Xs = self.x_scaler.transform(Xc)
        X_tensor = torch.from_numpy(Xs.astype(np.float32)).to(DEVICE)

        # Keep dropout ON at inference
        self.model.train()

        preds_scaled_mc = []
        with torch.no_grad():
            for _ in range(self.n_mc_samples):
                preds_scaled_mc.append(self.model(X_tensor).cpu().numpy().ravel())

        preds_scaled_mc = np.stack(preds_scaled_mc, axis=0)
        mean_scaled = preds_scaled_mc.mean(axis=0)
        std_scaled = preds_scaled_mc.std(axis=0)

        # RobustScaler inversion: y = mean_scaled * scale_ + center_
        scale_y = float(self.y_scaler.scale_[0])
        center_y = float(self.y_scaler.center_[0])

        mean = mean_scaled * scale_y + center_y
        if not return_std:
            return mean

        std = std_scaled * abs(scale_y)
        return mean, std


# ---------------------------------------------------------
# 4. Acquisition helpers
# ---------------------------------------------------------
def acquisition_pi_ei(mu: np.ndarray, sigma: np.ndarray, y_best: float, xi: float = 0.0):
    sigma = np.maximum(sigma, 1e-9)
    gamma = (mu - y_best - xi) / sigma
    pi = norm.cdf(gamma)
    ei = (mu - y_best - xi) * pi + sigma * norm.pdf(gamma)
    return pi, np.maximum(ei, 0.0)


# ---------------------------------------------------------
# 5. Week 8 proposer
#    Returns x_next rounded to 6 decimals for submission
# ---------------------------------------------------------
def propose_next_point_week8(
    surrogate: MCDropoutSurrogate,
    X_obs: np.ndarray,
    y_obs: np.ndarray,
    bounds: Tuple[np.ndarray, np.ndarray],
    random_state: int = 123,
    xi: float = 0.0025,
    min_dist: float = 0.010,
    repulse_len: float = 0.050,
    repulse_w: float = 0.35,
    use_thompson_shortlist: bool = True,
    shortlist_q: float = 0.02,
    enforce_rounded_nonduplicate: bool = True
) -> Dict:
    rng = np.random.RandomState(random_state)
    lower, upper = np.asarray(bounds[0], float), np.asarray(bounds[1], float)

    # Clip observed for stable distance + best selection
    Xc = clip_to_bounds(X_obs, lower, upper)

    best_idx = int(np.argmax(y_obs))
    y_best = float(y_obs[best_idx])
    x_best = Xc[best_idx]

    # Candidate generation (two-stage)
    n1, n2 = 80_000, 60_000
    sigma1, sigma2 = 0.12, 0.04

    X1 = x_best + rng.normal(0.0, sigma1, size=(n1, Xc.shape[1]))
    X2 = x_best + rng.normal(0.0, sigma2, size=(n2, Xc.shape[1]))
    Xcand = clip_to_bounds(np.vstack([X1, X2]), lower, upper)

    mu, sig = surrogate.predict(Xcand, lower, upper, return_std=True)
    pi, ei = acquisition_pi_ei(mu, sig, y_best=y_best, xi=xi)

    # Distances to observed points (hard constraint)
    dmat = np.linalg.norm(Xcand[:, None, :] - Xc[None, :, :], axis=2)
    dmin = dmat.min(axis=1)
    ok = dmin >= min_dist

    # Soft repulsion
    penalty = np.exp(-(dmin ** 2) / (2.0 * repulse_len ** 2))
    score = ei - repulse_w * penalty
    score = np.where(ok, score, -np.inf)

    # Relax once if needed
    if not np.isfinite(score).any():
        ok = dmin >= (0.5 * min_dist)
        score = np.where(ok, ei - repulse_w * penalty, -np.inf)

    finite = np.isfinite(score)
    q_thr = np.quantile(score[finite], 1.0 - shortlist_q)
    shortlist = np.where(score >= q_thr)[0]

    if use_thompson_shortlist and shortlist.size > 0:
        # Thompson-like selection within shortlist
        Xs = surrogate.x_scaler.transform(Xcand[shortlist])
        X_tensor = torch.from_numpy(Xs.astype(np.float32)).to(DEVICE)
        surrogate.model.train()
        with torch.no_grad():
            draw_scaled = surrogate.model(X_tensor).cpu().numpy().ravel()
        scale_y = float(surrogate.y_scaler.scale_[0])
        center_y = float(surrogate.y_scaler.center_[0])
        draw = draw_scaled * scale_y + center_y
        next_idx = int(shortlist[int(np.argmax(draw))])
        pick_mode = "Thompson-shortlist"
    else:
        next_idx = int(np.argmax(score))
        pick_mode = "Max(EI - repulsion)"

    next_x = Xcand[next_idx].copy()

    # ---- Round to 6 decimals for the SUBMISSION datapoint ----
    next_x_6 = round6(next_x)

    # Optional: ensure the rounded point is not a duplicate (after rounding)
    # If duplicate, pick next-best score until unique.
    if enforce_rounded_nonduplicate:
        X_obs_6 = round6(clip_to_bounds(X_obs, lower, upper))
        # Build a fast set of tuples
        seen = set(map(tuple, X_obs_6))
        if tuple(next_x_6) in seen:
            # Try next best among top scores
            order = np.argsort(score)[::-1]  # descending
            found = False
            for j in order[:5000]:  # safety cap
                cand6 = round6(Xcand[j])
                if tuple(cand6) not in seen:
                    next_idx = int(j)
                    next_x = Xcand[next_idx].copy()
                    next_x_6 = cand6
                    pick_mode += " + nondup-fallback"
                    found = True
                    break
            if not found:
                # last resort: keep original (still valid, but may duplicate)
                pick_mode += " + DUPLICATE-RO"
                next_x_6 = round6(next_x)

    next_mu = float(mu[next_idx])
    next_sig = float(sig[next_idx])

    # nearest neighbors for reasoning
    dists = np.linalg.norm(Xc - next_x, axis=1)
    nn = np.argsort(dists)[:3]

    reasoning = [
        f"• Bounds: lower={lower.round(3)}, upper={upper.round(3)}",
        f"• Current best: y_best={y_best:.6f} at x_best={round6(x_best)} (X clipped to bounds).",
        "• Week 8 strategy:",
        f"   - EI with xi={xi}",
        f"   - hard min distance={min_dist}",
        f"   - soft repulsion w={repulse_w}, len={repulse_len}",
        f"   - selection mode: {pick_mode}, shortlist_q={shortlist_q}",
        f"• Pick (raw): x_next={next_x}, μ={next_mu:.6f}, σ={next_sig:.6f}, PI={float(pi[next_idx]):.4f}, EI={float(ei[next_idx]):.6f}",
        f"• Pick (6dp): x_next_6dp={next_x_6}",
        "• Nearest tested points:"
    ]
    for i, idx in enumerate(nn, 1):
        reasoning.append(f"   #{i}: x={round6(X_obs[idx])}, y={float(y_obs[idx]):.6f}, dist={float(dists[idx]):.6f}")

    return dict(
        next_x=next_x_6,          # <-- FINAL next point in 6 decimals
        next_x_raw=next_x,        # kept for debugging
        pred_mean=next_mu,
        pred_std=next_sig,
        pi=float(pi[next_idx]),
        ei=float(ei[next_idx]),
        y_best=y_best,
        x_best=round6(X_obs[best_idx]),
        reasoning="\n".join(reasoning),
    )


# ---------------------------------------------------------
# 6. Hyperparameter tuning (Random Search + Successive Halving)
# ---------------------------------------------------------
def cv_mse_score(
    config: Dict,
    X: np.ndarray,
    y: np.ndarray,
    bounds: Tuple[np.ndarray, np.ndarray],
    k: int = 5,
    seed: int = 0
) -> float:
    kf = KFold(n_splits=k, shuffle=True, random_state=seed)
    mses = []
    lower, upper = bounds

    for tr_idx, va_idx in kf.split(X):
        Xtr, Xva = X[tr_idx], X[va_idx]
        ytr, yva = y[tr_idx], y[va_idx]

        surr = MCDropoutSurrogate(
            hidden_layer_sizes=config["hidden"],
            dropout=config["dropout"],
            n_epochs=config["epochs"],
            lr=config["lr"],
            weight_decay=config["weight_decay"],
            random_state=seed,
            n_mc_samples=config["n_mc_samples"],
            robust_y=True
        )

        surr.fit(Xtr, ytr, lower, upper)
        preds = surr.predict(Xva, lower, upper)
        mses.append(np.mean((preds - yva) ** 2))

    return float(np.mean(mses))


def tune_hyperparameters(
    X: np.ndarray,
    y: np.ndarray,
    bounds: Tuple[np.ndarray, np.ndarray],
    random_state: int = 8
) -> Dict:
    rng = np.random.RandomState(random_state)

    hidden_options = [(64, 32), (64, 64), (128, 64)]
    dropout_options = [0.10, 0.15, 0.20]  # keep >0 for MC Dropout
    lr_options = [5e-4, 1e-3, 2e-3]
    wd_options = [0.0, 1e-6, 1e-5]

    n_initial = 14
    configs = []
    for _ in range(n_initial):
        cfg = dict(
            hidden=hidden_options[rng.randint(len(hidden_options))],
            dropout=float(dropout_options[rng.randint(len(dropout_options))]),
            lr=float(lr_options[rng.randint(len(lr_options))]),
            weight_decay=float(wd_options[rng.randint(len(wd_options))]),
            n_mc_samples=int([192, 256][rng.randint(2)]),
        )
        configs.append(cfg)

    stage_epochs = [500, 1200]
    keep_fracs = [0.5, 0.4]

    best_overall = None

    for stage, (epochs, keep_frac) in enumerate(zip(stage_epochs, keep_fracs), start=1):
        scored = []
        for cfg in configs:
            cfg_stage = dict(cfg)
            cfg_stage["epochs"] = epochs
            mse = cv_mse_score(cfg_stage, X, y, bounds=bounds, k=5, seed=0)
            scored.append((mse, cfg_stage))

        scored.sort(key=lambda t: t[0])
        if best_overall is None or scored[0][0] < best_overall[0]:
            best_overall = scored[0]

        k_keep = max(4, int(len(scored) * keep_frac))
        configs = [cfg for _, cfg in scored[:k_keep]]

        print(f"\n--- WEEK 8 TUNING STAGE {stage} ---")
        print(f"epochs={epochs}, kept={k_keep}/{len(scored)}")
        print(f"best CV-MSE so far: {best_overall[0]:.6f}")
        print(f"best config so far: {best_overall[1]}")

    return dict(best_cv_mse=best_overall[0], best_config=best_overall[1])


# ---------------------------------------------------------
# 7. Main: tune -> fit -> propose next (6 decimals)
# ---------------------------------------------------------
def main():
    np.random.seed(0)
    torch.manual_seed(0)

    bounds = (np.zeros(3), np.ones(3))

    # Tune hyperparameters
    tuning = tune_hyperparameters(X_raw, y_raw, bounds=bounds, random_state=8)
    best_cfg = tuning["best_config"]

    # Fit best surrogate on all data
    surrogate = MCDropoutSurrogate(
        hidden_layer_sizes=best_cfg["hidden"],
        dropout=best_cfg["dropout"],
        n_epochs=best_cfg["epochs"],
        lr=best_cfg["lr"],
        weight_decay=best_cfg["weight_decay"],
        random_state=0,
        n_mc_samples=best_cfg["n_mc_samples"],
        robust_y=True
    )
    surrogate.fit(X_raw, y_raw, bounds[0], bounds[1])

    # Current best observed
    best_idx = int(np.argmax(y_raw))
    current_best_x = round6(X_raw[best_idx])
    current_best_y = float(y_raw[best_idx])

    # Propose next query within [0,1]^3
    suggestion = propose_next_point_week8(
        surrogate=surrogate,
        X_obs=X_raw,
        y_obs=y_raw,
        bounds=bounds,
        random_state=123,
        xi=0.0025,
        min_dist=0.010,
        repulse_len=0.050,
        repulse_w=0.35,
        use_thompson_shortlist=True,
        shortlist_q=0.02,
        enforce_rounded_nonduplicate=True
    )

    print("\n================================================")
    print("WEEK 8 FUNCTION 3 — NEXT POINT (6 DECIMALS)")
    print("================================================")
    print("Surrogate: mc_dropout (robust y scaling)")
    print(f"Best CV-MSE (lower is better): {tuning['best_cv_mse']:.6f}")
    print("Best tuned hyperparameters:")
    print(f"  hidden: {best_cfg['hidden']}")
    print(f"  dropout: {best_cfg['dropout']}")
    print(f"  lr: {best_cfg['lr']}")
    print(f"  weight_decay: {best_cfg['weight_decay']}")
    print(f"  n_mc_samples: {best_cfg['n_mc_samples']}")
    print(f"  epochs: {best_cfg['epochs']}")

    print("\n================================================")
    print("CURRENT BEST OBSERVED")
    print("================================================")
    print(f"x_best = {current_best_x}, y_best = {current_best_y:.6f}")

    print("\n================================================")
    print("RECOMMENDED NEXT POINT (rounded to 6 decimals)")
    print("================================================")
    # Print as 6-decimal fixed formatting
    x_next = suggestion["next_x"]
    print(f"x_next     = [{x_next[0]:.6f}, {x_next[1]:.6f}, {x_next[2]:.6f}]")
    print(f"μ(x_next)  = {suggestion['pred_mean']:.6f}")
    print(f"σ(x_next)  = {suggestion['pred_std']:.6f}")
    print(f"PI         = {suggestion['pi']:.4f}")
    print(f"EI         = {suggestion['ei']:.6f}")

    print("\n================================================")
    print("REASONING")
    print("================================================")
    print(suggestion["reasoning"])


if __name__ == "__main__":
    main()



--- WEEK 8 TUNING STAGE 1 ---
epochs=500, kept=7/14
best CV-MSE so far: 0.027575
best config so far: {'hidden': (64, 32), 'dropout': 0.15, 'lr': 0.001, 'weight_decay': 1e-06, 'n_mc_samples': 192, 'epochs': 500}

--- WEEK 8 TUNING STAGE 2 ---
epochs=1200, kept=4/7
best CV-MSE so far: 0.027546
best config so far: {'hidden': (64, 32), 'dropout': 0.15, 'lr': 0.001, 'weight_decay': 1e-06, 'n_mc_samples': 192, 'epochs': 1200}

WEEK 8 FUNCTION 3 — NEXT POINT (6 DECIMALS)
Surrogate: mc_dropout (robust y scaling)
Best CV-MSE (lower is better): 0.027546
Best tuned hyperparameters:
  hidden: (64, 32)
  dropout: 0.15
  lr: 0.001
  weight_decay: 1e-06
  n_mc_samples: 192
  epochs: 1200

CURRENT BEST OBSERVED
x_best = [0.403756 0.381706 0.489738], y_best = -0.009136

RECOMMENDED NEXT POINT (rounded to 6 decimals)
x_next     = [0.359927, 0.175969, 0.720956]
μ(x_next)  = -0.055263
σ(x_next)  = 0.011410
PI         = 0.0000
EI         = 0.000000

REASONING
• Bounds: lower=[0. 0. 0.], upper=[1. 1. 1.]
•